# 04 Automatic Glacier Delineation

Generate year-specific glacier masks from Sentinel-2 composites, GLIMS spatial priors, and Copernicus GLO-30 terrain constraints. This notebook submits Earth Engine Drive export tasks and writes a resumable manifest.

In [ ]:
from pathlib import Path
import sys

import geopandas as gpd

PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
sys.path.insert(0, str(PROJECT_ROOT))

from igcd.config import load_config
from igcd.ee_utils import authenticate_and_initialize
from igcd.glacier_delineation import export_delineation_for_inventory
from igcd.utils import setup_logging

config = load_config(PROJECT_ROOT / 'config' / 'config.json')
logger = setup_logging(config.paths['logs'])
authenticate_and_initialize(project=config.raw['earth_engine']['project'])

inventory_path = config.paths['processed'] / 'inventory' / 'glacier_inventory.geojson'
inventory = gpd.read_file(inventory_path)
max_glaciers = config.raw.get('processing', {}).get('max_glaciers_per_export_batch')
if max_glaciers is not None:
    inventory = inventory.head(int(max_glaciers)).copy()
inventory['geometry'] = inventory.geometry.apply(lambda geom: geom.__geo_interface__)
manifest = export_delineation_for_inventory(
    inventory,
    config,
    logger,
    config.paths['reports'] / 'glacier_delineation_manifest.csv',
    max_tasks=config.raw['export'].get('max_tasks_per_run')
)
manifest.head()

For fully adaptive local delineation after exports are downloaded, use `igcd.glacier_delineation.delineate_glacier_mask` with aligned Sentinel, DEM, slope, and rasterized-prior arrays. The Earth Engine export path uses the configured fallback thresholds for scalable server-side processing.